# Insider Threat Behavioral Intelligence System

## Model Training (Isolation Forest)

This notebook trains an Isolation Forest anomaly detection model using the engineered behavioral features extracted from the CERT insider threat dataset.

### Objective

The objective of this notebook is to build an unsupervised anomaly detection model that can identify suspicious insider behavior based on user activity features.

In [2]:
import pandas as pd
import joblib

from pathlib import Path

from sklearn.ensemble import IsolationForest

## Load Engineered Feature Dataset

This section loads the final engineered feature dataset generated during the feature engineering phase. This dataset will be used for training the Isolation Forest anomaly detection model.

In [3]:
PROJECT_ROOT = Path.cwd().parents[1]

DATA_PATH = PROJECT_ROOT / "datasets" / "processed" / "final_features.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset Shape:")
print(df.shape)

df.head()

Dataset Shape:
(265, 11)


,user,device_connections,emails_sent,files_accessed,websites_visited,logon_count,O,C,E,A,N
0,LRR0148,1569,0,907,57200,1378,21,19,14,14,29
1,HAH0760,1342,150,1900,10034,957,34,17,22,37,30
2,BIS0247,232,413,418,31140,1056,36,19,18,39,37
3,MAR0955,42,0,69,2963,604,36,44,23,44,25
4,OCS0865,1310,725,873,56052,1005,29,24,43,35,32


## Dataset Verification

This section verifies that the engineered dataset has been loaded correctly before model training.

In [4]:
print("Columns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

Columns:
['user', 'device_connections', 'emails_sent', 'files_accessed', 'websites_visited', 'logon_count', 'O', 'C', 'E', 'A', 'N']

Data Types:
user                    str
device_connections    int64
emails_sent           int64
files_accessed        int64
websites_visited      int64
logon_count           int64
O                     int64
C                     int64
E                     int64
A                     int64
N                     int64
dtype: object

Missing Values:
user                  0
device_connections    0
emails_sent           0
files_accessed        0
websites_visited      0
logon_count           0
O                     0
C                     0
E                     0
A                     0
N                     0
dtype: int64


## Feature Selection

This section prepares the feature matrix for model training. The user identifier column is removed because it is not a behavioral feature and should not influence anomaly detection.

In [5]:
# Separate User IDs

users = df["user"]

# Prepare feature matrix

X = df.drop(columns=["user"])

print("Feature Matrix Shape:")
print(X.shape)

X.head()

Feature Matrix Shape:
(265, 10)


,device_connections,emails_sent,files_accessed,websites_visited,logon_count,O,C,E,A,N
0,1569,0,907,57200,1378,21,19,14,14,29
1,1342,150,1900,10034,957,34,17,22,37,30
2,232,413,418,31140,1056,36,19,18,39,37
3,42,0,69,2963,604,36,44,23,44,25
4,1310,725,873,56052,1005,29,24,43,35,32


## Train Isolation Forest Model

This section trains the Isolation Forest algorithm using the engineered behavioral features. The model learns the normal behavior of employees and identifies anomalous users without requiring labeled data.

In [6]:
model = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42
)

model.fit(X)

print("Model trained successfully!")

Model trained successfully!


## Predict Insider Threats

This section predicts whether each employee exhibits normal or anomalous behavior using the trained Isolation Forest model.

In [7]:
predictions = model.predict(X)

print(predictions[:20])

[ 1  1  1  1  1  1  1  1  1 -1  1 -1  1  1  1  1  1  1  1  1]


## Attach Prediction Results

This section appends the predicted anomaly labels to the dataset for further analysis.

In [8]:
df["prediction"] = predictions

df.head()

,user,device_connections,emails_sent,files_accessed,websites_visited,logon_count,O,C,E,A,N,prediction
0,LRR0148,1569,0,907,57200,1378,21,19,14,14,29,1
1,HAH0760,1342,150,1900,10034,957,34,17,22,37,30,1
2,BIS0247,232,413,418,31140,1056,36,19,18,39,37,1
3,MAR0955,42,0,69,2963,604,36,44,23,44,25,1
4,OCS0865,1310,725,873,56052,1005,29,24,43,35,32,1


In [9]:
print("Prediction Counts:")

print(df["prediction"].value_counts())

Prediction Counts:
prediction
 1    251
-1     14
Name: count, dtype: int64


## Save Trained Model

The trained Isolation Forest model is saved for future inference. The backend and streaming modules will load this model to perform real-time insider threat prediction without retraining.

In [10]:
import os
import joblib
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]

MODEL_PATH = PROJECT_ROOT / "ml" / "models"

MODEL_PATH.mkdir(exist_ok=True)

joblib.dump(
    model,
    MODEL_PATH / "isolation_forest.pkl"
)

print("Model saved successfully!")
print(MODEL_PATH / "isolation_forest.pkl")

Model saved successfully!
c:\Projects\InsiderThreat\Insider-Threat-Behavioral-Intelligence-System\ml\models\isolation_forest.pkl


## Save Prediction Results

This section stores the anomaly prediction results for analysis and visualization.

In [11]:
OUTPUT_PATH = PROJECT_ROOT / "datasets" / "processed"

df.to_csv(
    OUTPUT_PATH / "prediction_results.csv",
    index=False
)

print("Prediction results saved!")

Prediction results saved!
